In [ ]:
def MA_cross_sweep_test(self, symbol, start, end, timeframe='m15', short_window=range(2, 100, 2), long_window=range(20, 1000, 2), optimize_on='sharpe_ratio', max_workers=None, debug_single_thread=False, strat_mode='volume_0'):
        start_to_work = time.time()
        # Validate inputs
        if optimize_on not in ['sharpe_ratio', 'returns']:
            raise ValueError("optimize_on must be 'sharpe' or 'returns'.")

        # Set max_workers
        max_workers = max_workers or min(32, os.cpu_count() or 4)
        print(f"Using {max_workers} workers for threading" if not debug_single_thread else "Running in single-threaded mode")

        # Initialize thread-safe results list
        results = []
        lock = Lock()

        # Function to evaluate a single MA pair
        def evaluate_pair(pair):
            short, long = pair
            if short >= long:
                print(f"Skipping invalid pair: short={short}, long={long}")
                return None
            try:
                print(f"Evaluating pair: short={short}, long={long}")
                returns, sharpe_ratio, final_returns, max_dd = self.MA_cross(symbol, start, end, short, long, timeframe=timeframe, strat_mode=strat_mode)

                if not (np.isfinite(sharpe_ratio) and np.isfinite(final_returns) and np.isfinite(max_dd)):
                    print(f"Non-finite results for pair ({short}, {long}): sharpe={sharpe_ratio}, returns={final_returns}, max_dd={max_dd}")
                    return None
                print(f"Valid pair ({short}, {long}): sharpe={sharpe_ratio:.4f}, returns={final_returns:.4f}, max_dd={max_dd:.4f}")
                result = {
                    'returns_df': returns,
                    'short_window': short,
                    'long_window': long,
                    'sharpe_ratio': sharpe_ratio,
                    'final_returns': final_returns,
                    'max_drawdown': max_dd
                }
                return result
            except Exception as e:
                print(f"Error for pair ({short}, {long}): {str(e)}")
                return None

        # Generate valid pairs
        pairs = [(short, long) for short, long in product(short_window, long_window) if short < long - 5]
        print(f"Total pairs to evaluate: {len(pairs)}")

        # Pre-check data
        try:
            data = pd.DataFrame(self.account.get_data_from_range(symbol, 'm15', start, end))
            if data.empty or not all(col in data for col in ['high', 'low', 'close', 'tick_volume']):
                print("Invalid or empty data returned")
                return {
                    'results': pd.DataFrame(),
                    'best_pair': None,
                    'best_metric': None,
                    'concatenated_returns': pd.DataFrame()
                }
            if data['tick_volume'].eq(0).any() or data[['high', 'low', 'close', 'tick_volume']].isna().any().any():
                print("Data contains NaNs or zero tick_volume")
                return {
                    'results': pd.DataFrame(),
                    'best_pair': None,
                    'best_metric': None,
                    'concatenated_returns': pd.DataFrame()
                }
            if len(data) < max(long_window):
                print(f"Insufficient data points: {len(data)} < {max(long_window)}")
                return {
                    'results': pd.DataFrame(),
                    'best_pair': None,
                    'best_metric': None,
                    'concatenated_returns': pd.DataFrame()
                }
            print(f"Data loaded successfully: {len(data)} rows")
        except Exception as e:
            print(f"Failed to load data: {str(e)}")
            return {
                'results': pd.DataFrame(),
                'best_pair': None,
                'best_metric': None,
                'concatenated_returns': pd.DataFrame()
            }

        # Run evaluations
        if debug_single_thread:
            results_list = [evaluate_pair(pair) for pair in tqdm(pairs, desc="Sweeping MA pairs")]
        else:
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                results_list = list(tqdm(executor.map(evaluate_pair, pairs), total=len(pairs), desc="Sweeping MA pairs"))
                # print (results_list)
        # Collect valid results
        with lock:
            results.extend([r for r in results_list if r is not None])

        # Convert results to DataFrame for analysis
        results_df = pd.DataFrame([{
            'short_window': r['short_window'],
            'long_window': r['long_window'],
            'sharpe_ratio': r['sharpe_ratio'],
            'final_returns': r['final_returns'],
            'max_drawdown': r['max_drawdown']
        } for r in results])

        if results_df.empty:
            print("No valid results to analyze")
            return {
                'results': results_df,
                'best_pair': None,
                'best_metric': None,
                'concatenated_returns': pd.DataFrame()
            }

        # Perform K-means clustering
        from sklearn.cluster import KMeans
        from sklearn.preprocessing import StandardScaler
        import matplotlib.pyplot as plt

        # Prepare features for clustering
        features = results_df[['sharpe_ratio', 'final_returns', 'max_drawdown']]
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(features)

        # Apply K-means clustering
        n_clusters = 5
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        results_df['cluster'] = kmeans.fit_predict(scaled_features)

        # Create scatter plot
        plt.figure(figsize=(10, 6))
        scatter = plt.scatter(results_df['short_window'], results_df['long_window'],
                            c=results_df['cluster'], cmap='viridis',
                            s=50, alpha=0.6)
        plt.colorbar(scatter, label='Cluster')
        plt.xlabel('Short Window')
        plt.ylabel('Long Window')
        plt.title(f'MA Pairs Clustering (Sharpe, Returns, Drawdown) - {symbol}')

        # Add annotations for best performing pairs
        best_idx = results_df[optimize_on].idxmax()
        best_pair = results_df.loc[best_idx]
        plt.annotate(f'Best {optimize_on}\n({best_pair["short_window"]}, {best_pair["long_window"]})\n{optimize_on}: {best_pair[optimize_on]:.2f}',
                    (best_pair['short_window'], best_pair['long_window']),
                    xytext=(5, 5), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))

        plt.tight_layout()
        plt.show()

        end = time.time()
        print(f'Processing time is {end - start_to_work}')

        # Find best pair
        best_idx = results_df[optimize_on].idxmax()
        best_pair = (results_df.loc[best_idx, 'short_window'],
                    results_df.loc[best_idx, 'long_window'])
        best_metric = results_df.loc[best_idx, optimize_on]

        return {
            'results': results_df,
            'best_pair': best_pair,
            'best_metric': best_metric,
            'concatenated_returns': pd.DataFrame()  # Empty since we're not combining returns anymore
        }